# Installation des dépendances (à exécuter une seule fois)


In [1]:
!pip install langchain langchain-community chromadb openai
!pip install -q langchain_huggingface sentence_transformers --no-deps
!pip install -U langchain-huggingface
!pip install -q bert-score


# Imports globaux


In [2]:
import csv
import glob
import json
import random
import re
import shutil
import statistics
from datetime import datetime
from pathlib import Path

import torch
from bert_score import score as bert_score_fn
from langchain_classic.memory import ConversationBufferMemory
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFacePipeline
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline


# Configuration globale


In [3]:
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
RUN_CHAT = False

MAX_NEW_TOKENS_EVAL = 128
MAX_NEW_TOKENS_CHAT = 256 # 256 tokens maximum afin d’optimiser le temps d’inférence
# et prévenir les dépassements mémoire (OOM) observés avec des limites plus élevées.

K = 4  # nombre de documents finaux retournés par le retrieval

CHUNK_SIZE = 600
CHUNK_OVERLAP = 60

EVAL_MAX_QUERIES = None  # ex: 120 pour tests rapides, None pour toutes les requêtes
EVAL_RANDOM_SEED = 42 # Sert à rendre l'échantillonnage des requêtes reproductible quand EVAL_MAX_QUERIES est défini.

DATASET_FOLDER_PATH = Path("dataset_eval")
BERT_THRESHOLD = 0.60  # seuil accepté pour le F1 BERTScore (60%)
LANG = "fr"

CHROMA_DB_PATH = Path("./chroma_db")
CHROMA_EVAL_DB_PATH = Path("./chroma_eval_db")

RESULTS_FOLDER_PATH = DATASET_FOLDER_PATH / "result"
RESULTS_FOLDER_PATH.mkdir(parents=True, exist_ok=True)
RESULTS_LAST_FOLDER_PATH = RESULTS_FOLDER_PATH / "last"
RESULTS_LAST_FOLDER_PATH.mkdir(parents=True, exist_ok=True)


# 1) Préparation du retrieval
## 1.1) Ingestion des PDF


In [4]:
# Chargement des PDF du dossier DIC
documents = []

# Chaque page devient un objet Document contenant :
# 	• 	page_content → le texte
# 	•	metadata → source, page, auteur, etc.

for file in glob.glob("DIC/*.pdf"):
    try:
        loader = PyPDFLoader(file)  # Retourne une liste de document (un pour chaque page)
        documents += loader.load()
    except Exception as e:
        print(f"Erreur survenue pour le fichier '{file}' : {e}")


## 1.1) Vérification du chargement


In [5]:
# Première page du premier document PDF
documents[0]


Document(metadata={'producer': 'Actuate PDF Writer (Low Resolution) 2.1', 'creator': 'Actuate e.Reports', 'creationdate': '2022-07-29T08:11:45+01:00', 'title': '', 'subject': '', 'author': 'IDS GmbH - Analysis and Reporting Services', 'keywords': 'FR0010032326 (22.08.2022)', 'source': 'DIC/Allianz.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='\x00I\x00n\x00f\x00o\x00r\x00m\x00a\x00t\x00i\x00o\x00n\x00s\x00 \x00c\x00l\x00Ø\x00s\x00 \x00p\x00o\x00u\x00r\x00 \x00l \x19\x00i\x00n\x00v\x00e\x00s\x00t\x00i\x00s\x00s\x00e\x00u\x00r\n\x00C\x00e\x00 \x00d\x00o\x00c\x00u\x00m\x00e\x00n\x00t\x00 \x00f\x00o\x00u\x00r\x00n\x00i\x00t\x00 \x00d\x00e\x00s\x00 \x00i\x00n\x00f\x00o\x00r\x00m\x00a\x00t\x00i\x00o\x00n\x00s\x00 \x00e\x00s\x00s\x00e\x00n\x00t\x00i\x00e\x00l\x00l\x00e\x00s\x00 \x00a\x00u\x00x\x00 \x00i\x00n\x00v\x00e\x00s\x00t\x00i\x00s\x00s\x00e\x00u\x00r\x00s\x00 \x00d\x00e\x00 \x00c\x00e\x00t\x00 \x00O\x00P\x00C\x00V\x00M\x00.\x00 \x00I\x00l\x00 \x00n\x00e\x00 \x00s

## 1.2) Découpage en chunks


In [6]:
# Initialisation du séparateur de texte
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,  # Taille maximale des morceaux de texte
    chunk_overlap=CHUNK_OVERLAP,  # Chevauchement entre les morceaux pour garder le contexte
    length_function=len,  # Fonction pour calculer la longueur des morceaux
    separators=["\n\n", "\n"]  # Séparateurs utilisés pour diviser le texte en morceaux
)

# Division du document en morceaux (chunks)
chunks = text_splitter.split_documents(documents=documents)

print(f"{len(chunks)} chunks ont été créés par le splitter à partir des documents PDF du dossier DIC.")


1721 chunks ont été créés par le splitter à partir des documents PDF du dossier DIC.


## 1.3) Embeddings


In [7]:
# Choix du modèle d'embedding
# On normalise les embeddings pour améliorer la stabilité de la similarité cosinus.
# Modèle open-source multilingue adapté au français, exécutable localement sans API externe.
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME, encode_kwargs={"normalize_embeddings" : True})


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## 1.4) Construction de la base vectorielle (Chroma)


In [8]:
# Nettoyage de la base `chroma_db` si elle existe
persist_path = CHROMA_DB_PATH

# Supprime récursivement le dossier persist_path et tout son contenu (fichiers + sous-dossiers)
if persist_path.exists():
    shutil.rmtree(persist_path)
persist_path.mkdir(parents=True, exist_ok=True)

# Créer la base Chroma (locale)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(persist_path),  # stockage local
    collection_name="col_production"
)


## 1.5) Vérification rapide du retrieval


In [9]:
query = "Que montre le scénario de tensions ?"
results = vectorstore.similarity_search(query)

for r in results:
    print(r.page_content)


Les scénarios défavorable, intermédiaire et favorable présentés représentent des exemples utilisant les meilleures et pires per formances, ainsi que la performance 
moyenne de l’indice de référence approprié au cours des 10 dernières années.  
Les marchés pourraient évoluer très différemment à l’avenir. Le scénario de tension montre ce que vous pourriez obtenir dans des situations de marché extrêmes. 
Il n'est pas facile de sortir de ce produit. Si vous sortez de l’investissement avant la fin de période de détention recommandée, aucune garantie ne vous est donnée.
Les scénarios défavorable, intermédiaire et favorable présentés représentent des exemples utilisant les meilleures et pires per formances, ainsi que la performance 
moyenne de l’indice de référence approprié au cours des 10 dernières années.  
Les marchés pourraient évoluer très différemment à l’avenir. Le scénario de tension montre ce que vous pourriez obtenir dans des situations de marché extrêmes. 
Il n'est pas facile de s

# 2) Initialisation du modèle de génération


In [10]:
# Mistral 7B : bon compromis qualité / VRAM / latence pour ce RAG.

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16
)

print(model.config._name_or_path)

generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    do_sample=False,
    return_full_text=False
)

# Création d'une instance de HuggingFacePipeline à partir de l'identifiant du modèle spécifié (LangChain wrapper)
llm = HuggingFacePipeline(pipeline=generation_pipeline)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


mistralai/Mistral-7B-Instruct-v0.2


## 2.1) Pipeline RAG (mémoire, prompt, génération)


In [11]:
memory = ConversationBufferMemory(
    memory_key="history",
    return_messages=False
)

prompt_template = """Tu es un assistant spécialisé en analyse de documents d’informations clés (DIC) pour un département ALM d’assurance vie. 
Réponds uniquement à partir du contexte fourni.
Tu ne dois jamais utiliser tes connaissances internes.
Tu dois obligatoirement citer la source exacte du document utilisé pour formuler la réponse.
Si aucune source n’est trouvée dans le contexte fourni,
indique explicitement :
"Information non disponible dans les DIC fournis."

Historique de la conversation :
{history}

Contexte :
{context}

Question :
{question}

Réponse (réponds uniquement à la question posée, sans ajouter de nouvelle question) :
"""

def has_valid_source_citation(text: str, use_eval: bool = False) -> bool:
    if not text or not text.strip():
        return False

    if use_eval:
        # En évaluation, on accepte un format de citation minimal basé sur l'UUID
        pattern_eval = r"sources?\s*:\s*(?:[-*]\s*)?(?:[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}|[\r\n\s-]+[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12})"
        return re.search(pattern_eval, text, flags=re.IGNORECASE) is not None

    # En chat/prod, on attend un format plus strict: "Source: ... - Page <nombre>"
    pattern_prod = r"sources?\s*:\s*.+?\s*-\s*page\s*\d+"
    return re.search(pattern_prod, text, flags=re.IGNORECASE | re.DOTALL) is not None

def rag_pipeline(query, use_eval=False):
    # Choix du vectorstore
    current_vectorstore = vectorstore_eval if use_eval else vectorstore

     # Récupérer l'historique (PAS de memory en mode évaluation)
    if use_eval:
        history = ""
    else:
        history = memory.load_memory_variables({})["history"]
    
    # Limitation à K documents récupérés pour contrôler la longueur du prompt et stabiliser le pipeline RAG.
    retrieved_docs = current_vectorstore.similarity_search(query, k=K)

    if use_eval:
        context = "\n\n".join(
            f"Source: {doc.metadata.get('uuid', 'inconnue')}\n{doc.page_content}"
            for doc in retrieved_docs
        )
    else:
        context = "\n\n".join(
            f"Source: {doc.metadata.get('source', 'Inconnue')} - Page {doc.metadata.get('page', '?')}\n{doc.page_content}"
            for doc in retrieved_docs
        )
    
    # Injection de l'historique et des documents (avec leurs metadata) dans le prompt
    prompt = prompt_template.format(
        history=history,
        question=query,
        context=context
    )
    # Choix de la longueur de génération selon le mode
    max_new_tokens = MAX_NEW_TOKENS_EVAL if use_eval else MAX_NEW_TOKENS_CHAT
    llm.pipeline.max_new_tokens = max_new_tokens

    response = llm.invoke(prompt, stop=["Question:"])
    response_str = str(response)

    if not has_valid_source_citation(response_str, use_eval=use_eval):
        response_str += "\n\n[AVERTISSEMENT] Réponse non conforme: source manquante."

    # Sauvegarde dans la mémoire (sauf en mode evaluation)
    if not use_eval:
        memory.save_context({"input": query}, {"output": response_str})

    return response_str


/tmp/ipykernel_750/840554775.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


## 2.2) Test d'une requête


In [12]:
query = """
Quel est l'objectif de gestion du FCP décrit dans le document ? ?
"""

response = rag_pipeline(query)
print(response)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



L'objectif de gestion du FCP décrit dans les documents est de rechercher une performance sur la durée de placement recommandée supérieure à 5 ans, en s'exposant aux marchés financiers actions internationales, pays émergents inclus, tout en tenant compte des enjeux de développement durable. Les décisions d'investissement intègrent à la fois des critères financiers et extra-financiers, et la prise en compte des critères de responsabilité environnementale, sociale et de governance est mentionnée dans trois des documents.

Sources : DIC/FR0007050570_DIC_FR_20230630.pdf, DIC/FR001400BQ78_DIC_FR_20230630.pdf, DIC/FR0007040373_DIC_FR_20230630.pdf.

[AVERTISSEMENT] Réponse non conforme: source manquante.


## 2.3) Boucle de chat interactive


In [13]:
def chat():
    print("Assistant DIC prêt. Tape 'exit' pour quitter.\n")
    
    while True:
        user_input = input("Vous : ")
        
        if user_input.lower() in ["exit", "quit"]:
            print("Fin de la conversation.")
            break
        
        response = rag_pipeline(user_input)
        print("\nAssistant :", response, "\n")


## 2.4) Exécution conditionnelle du chat


In [14]:
memory.clear()
if RUN_CHAT:
    chat()


# 3) Évaluation
## 3.1) Re-création de la base de test à partir de `corpus.json`

In [15]:
# Nettoyage de la base chroma_eval_db si elle existe
persist_path = CHROMA_EVAL_DB_PATH
if persist_path.exists():
    shutil.rmtree(persist_path)
persist_path.mkdir(parents=True, exist_ok=True)

# Chargement du corpus
with open(DATASET_FOLDER_PATH / "corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

print(f"Nombre de chunks : {len(corpus)}")

# Transformation en Documents
documents = [
    Document(page_content=text, metadata={"uuid": uid})
    for uid, text in corpus.items()
]

print(f"{len(documents)} Documents prêts.")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME
)

# Vectorstore
vectorstore_eval = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=str(persist_path),
    collection_name="col_evaluation"
)

print("Base d’évaluation construite avec succès.")


Nombre de chunks : 250
250 Documents prêts.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Base d’évaluation construite avec succès.


## 3.2) Évaluation du retrieval (Recall@k) et de la génération (F1 BERTScore)


In [16]:
# Chargement des fichiers d'évaluation
with open(DATASET_FOLDER_PATH / "queries.json", "r", encoding="utf-8") as f:
    queries = json.load(f)  # dict uuid -> query_text

with open(DATASET_FOLDER_PATH / "answers.json", "r", encoding="utf-8") as f:
    answers = json.load(f)  # dict uuid -> expected_answer_text

with open(DATASET_FOLDER_PATH / "relevant_docs.json", "r", encoding="utf-8") as f:
    relevant_docs = json.load(f)  # dict uuid -> [list of relevant doc uuids]

results = []

# Boucle d'évaluation sur les UUID présents dans `queries`, `answers` et `relevant_docs`
common_uuids = sorted(
    set(queries.keys())
    .intersection(set(answers.keys()))
    .intersection(set(relevant_docs.keys()))
)

if EVAL_MAX_QUERIES is not None:
    rng = random.Random(EVAL_RANDOM_SEED)
    common_uuids = rng.sample(common_uuids, k=min(EVAL_MAX_QUERIES, len(common_uuids)))
    common_uuids = sorted(common_uuids)
print(f"Evaluation de {len(common_uuids)} requêtes (K={K}, max={EVAL_MAX_QUERIES}) — device torch: {torch.cuda.is_available() and 'cuda' or 'cpu'}")

# tqdm sert à visualiser la progression d’une boucle.
for uid in tqdm(common_uuids, desc="Evaluation RAG"):
    query = queries[uid].strip()
    ref_answer = answers[uid].strip()

    # 1) Générer la réponse via la pipeline d'évaluation
    try:
        raw_gen_answer = rag_pipeline(query, use_eval=True)
        # Si rag_pipeline renvoie un objet complexe, extraire string :
        if isinstance(raw_gen_answer, (list, tuple)):
            gen_answer = " ".join(str(x) for x in raw_gen_answer)
        else:
            gen_answer = str(raw_gen_answer)
    except Exception as e:
        gen_answer = ""
        print(f"[ERROR] génération pour {uid} a levé : {e}")

    # 2) retrieval (même comportement que dans la pipeline)
    retrieved_docs = vectorstore_eval.similarity_search(query, k=K)

    # Extraire identifiants/uuids des docs récupérés — on cherche un champ 'uuid' ou 'source' dans metadata
    # Chaque chunk indexé dans Chroma a son uuid du corpus.json.
    retrieved_uuids = [doc.metadata["uuid"] for doc in retrieved_docs]

    # 3) comparer aux relevant_docs attendus (liste)
    expected_uuids = relevant_docs[uid]
    # Le recall mesure la proportion des documents réellement pertinents
    # (définis dans relevant_docs.json) qui ont été correctement récupérés
    # par le moteur de recherche du pipeline RAG.
    if len(expected_uuids) == 0:
        recall_at_k = None
    else:
        found = sum(1 for e in expected_uuids if str(e) in retrieved_uuids)
        recall_at_k = found / len(expected_uuids)

    results.append({
        "uuid": uid,
        "query": query,
        "gen": gen_answer,
        "ref": ref_answer,
        "retrieved_uuids": retrieved_uuids,
        "expected_uuids": expected_uuids,
        "recall_at_k": recall_at_k,
        "has_source_citation": has_valid_source_citation(gen_answer, use_eval=True),
    })

# 4) Calcul du BERTScore (en batch pour accélérer)
# Le BERTScore évalue la proximité sémantique entre deux textes à l’aide d’un modèle BERT.
print("Calcul du BERTScore (F1) — ceci peut prendre quelques secondes/minutes selon la GPU/CPU...")
device = "cuda" if torch.cuda.is_available() else "cpu"

gen_answers = [r["gen"] for r in results]
ref_answers  = [r["ref"] for r in results]

# Le BERTScore utilisé dans cette évaluation correspond au F1,
# qui combine la précision et le rappel sémantiques entre la réponse générée et la référence.
P, R, F1 = bert_score_fn(cands=gen_answers,
                         refs=ref_answers,
                         lang=LANG,
                         model_type=None,  # laisser bert-score choisir le modèle adapté à la langue
                         device=device,
                         verbose=True)

# bert_score renvoie tensors — convertir en float
F1_list = [float(x) for x in F1]

# Remplir les résultats avec le F1
for i, uid in enumerate(common_uuids):
    results[i]["bert_f1"] = F1_list[i]

# 5) métriques agrégées
valid_f1s = [v for v in F1_list if not (v is None or (isinstance(v, float) and (v != v)))]
mean_f1 = statistics.mean(valid_f1s) if valid_f1s else 0.0
# %des query qui ont F1 > 60%
pct_above_threshold = 100.0 * sum(1 for v in valid_f1s if v >= BERT_THRESHOLD) / len(valid_f1s) if valid_f1s else 0.0

# calcul de la moyenne du recall
recalls = [r["recall_at_k"] for r in results if r["recall_at_k"] is not None]
mean_recall_at_k = statistics.mean(recalls) if recalls else None


Evaluation de 619 requêtes (K=4, max=None) — device torch: cuda


Evaluation RAG:   0%|          | 0/619 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Evaluation RAG:   0%|          | 1/619 [00:01<18:42,  1.82s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Evaluation RAG:   0%|          | 2/619 [00:03<16:33,  1.61s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Plea

Calcul du BERTScore (F1) — ceci peut prendre quelques secondes/minutes selon la GPU/CPU...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/16 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/10 [00:00<?, ?it/s]

done in 2.11 seconds, 293.13 sentences/sec


## 3.3) Résultats agrégés et export principal


In [17]:
print("\n--- Résultats agrégés ---")
print(f"Nombre de requêtes: {len(common_uuids)}")
print(f"Mean BERT-F1 : {mean_f1:.4f}")
print(f"% exemples avec BERT-F1 >= {BERT_THRESHOLD*100:.0f}% : {pct_above_threshold:.2f}%")
print(f"Mean recall@{K} (sur les requêtes ayant des relevant_docs) : {mean_recall_at_k if mean_recall_at_k is not None else 'N/A'}")

# Sauvegarde des résultats détaillés en CSV
OUT = RESULTS_LAST_FOLDER_PATH / "evaluation_results.csv"
with open(OUT, "w", encoding="utf-8", newline="") as csvfile:
    fieldnames = ["uuid","query","gen","ref","bert_f1","retrieved_uuids","expected_uuids","recall_at_k","has_source_citation"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for r in results:
        writer.writerow({
            "uuid": r["uuid"],
            "query": r["query"],
            "gen": r["gen"],
            "ref": r["ref"],
            "bert_f1": r.get("bert_f1"),
            "retrieved_uuids": "|".join(r["retrieved_uuids"]),
            "expected_uuids": "|".join([str(x) for x in r["expected_uuids"]]),
            "recall_at_k": r["recall_at_k"],
            "has_source_citation": r.get("has_source_citation")
        })

print(f"Résultats sauvegardés dans : {OUT}")

print("\n--- DEBUG RETRIEVAL (10 premiers exemples) ---")

for r in results[:10]:
    print("Expected:", r["expected_uuids"])
    print("Retrieved:", r["retrieved_uuids"])
    print()


ERRORS_OUT = RESULTS_LAST_FOLDER_PATH / "errors.json"
errors = [
    {
        "uuid": r["uuid"],
        "query": r["query"],
        "bert_f1": r.get("bert_f1"),
        "recall_at_k": r.get("recall_at_k"),
        "expected_uuids": r.get("expected_uuids"),
        "retrieved_uuids": r.get("retrieved_uuids"),
        "has_source_citation": r.get("has_source_citation"),
    }
    for r in results
    if (not r.get("gen", "").strip())
    or (r.get("bert_f1") is not None and r["bert_f1"] < BERT_THRESHOLD)
    or (r.get("recall_at_k") == 0)
    or (not r.get("has_source_citation", False))
]

with open(ERRORS_OUT, "w", encoding="utf-8") as f:
    json.dump(errors, f, ensure_ascii=False, indent=2)

print(f"Errors sauvegardés dans : {ERRORS_OUT} ({len(errors)} cas)")



--- Résultats agrégés ---
Nombre de requêtes: 619
Mean BERT-F1 : 0.6554
% exemples avec BERT-F1 >= 60% : 71.08%
Mean recall@4 (sur les requêtes ayant des relevant_docs) : 0.5735056542810986
Résultats sauvegardés dans : dataset_eval/result/last/evaluation_results.csv

--- DEBUG RETRIEVAL (10 premiers exemples) ---
Expected: ['9b1a4ec9-ef38-4e4c-aa7a-9e29d78aee12']
Retrieved: ['9b1a4ec9-ef38-4e4c-aa7a-9e29d78aee12', '69ee36e1-a8db-4228-a0f6-e53c269d44a0', '4f3683ac-486b-4481-9848-3d528cf31cef', 'd02d3cd7-1985-40a5-b18f-adec2cad376b']

Expected: ['6e774a8c-6ae7-49ab-b35d-5e991146b905']
Retrieved: ['86cf988f-69d7-48cf-9d56-44ad8e68ad86', '375f41cc-141c-4dd5-ade4-bb9b39d3dfe0', 'a443ec75-7fe2-4c25-85e4-3b088adb0bec', '35e9b882-a723-4078-98ee-3b587fedf04a']

Expected: ['a661416b-be2d-4141-8985-5fb6bfc902c0']
Retrieved: ['7d87da2b-f76f-4a60-b6dc-184d7f934824', 'e56b2960-c34f-499c-9806-facc0c834151', 'aa6bd01b-b7c4-4af4-9e6f-8a9c7ad681d8', 'bc8488cc-3d76-4e21-9226-68d382a50a0c']

Expected: ['

## 3.4) Export versionné (timestamp)


In [18]:
safe_model_name = re.sub(r"[^\w\-]", "_", MODEL_ID.split("/")[-1])

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

SUMMARY_OUT = RESULTS_LAST_FOLDER_PATH / f"evaluation_summary_{safe_model_name}_k{K}_thr{int(BERT_THRESHOLD*100)}_{timestamp}.csv"
DETAIL_OUT = RESULTS_LAST_FOLDER_PATH / f"evaluation_results_{safe_model_name}_k{K}_thr{int(BERT_THRESHOLD*100)}_{timestamp}.csv"

with open(DETAIL_OUT, "w", encoding="utf-8", newline="") as csvfile:
    fieldnames = [
        "uuid","query","gen","ref","bert_f1",
        "retrieved_uuids","expected_uuids","recall_at_k","has_source_citation"
    ]
    
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    
    for r in results:
        writer.writerow({
            "uuid": r["uuid"],
            "query": r["query"],
            "gen": r["gen"],
            "ref": r["ref"],
            "bert_f1": r.get("bert_f1"),
            "retrieved_uuids": "|".join(r["retrieved_uuids"]),
            "expected_uuids": "|".join([str(x) for x in r["expected_uuids"]]),
            "recall_at_k": r["recall_at_k"],
            "has_source_citation": r.get("has_source_citation")
        })

with open(SUMMARY_OUT, "w", encoding="utf-8", newline="") as csvfile:
    fieldnames = [
        "model_name",
        "k",
        "threshold",
        "nb_queries",
        "mean_bert_f1",
        "pct_above_threshold",
        f"mean_recall_at_{K}"
    ]
    
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    
    writer.writerow({
        "model_name": MODEL_ID,
        "k": K,
        "threshold": BERT_THRESHOLD,
        "nb_queries": len(common_uuids),
        "mean_bert_f1": round(mean_f1, 4),
        "pct_above_threshold": round(pct_above_threshold, 2),
        f"mean_recall_at_{K}": round(mean_recall_at_k, 4) if mean_recall_at_k is not None else "N/A"
    })

print(f"Détail sauvegardé dans : {DETAIL_OUT}")
print(f"Résumé sauvegardé dans : {SUMMARY_OUT}")


Détail sauvegardé dans : dataset_eval/result/last/evaluation_results_Mistral-7B-Instruct-v0_2_k4_thr60_20260322_142023.csv
Résumé sauvegardé dans : dataset_eval/result/last/evaluation_summary_Mistral-7B-Instruct-v0_2_k4_thr60_20260322_142023.csv
